# 03 · Indicadores para Producers Direct

**Objetivo.** Convertir la base limpia en un conjunto pequeño de indicadores, cada uno con su definición, su fórmula, su denominador y la decisión que apoya. Son los mismos indicadores que muestra el tablero.

**Enfoque.** Se parte de la necesidad del cliente (fase de comprensión del negocio en CRISP-DM) y de ahí se eligen los indicadores, no al revés: un indicador que no apoya una decisión no se incluye.

| | Archivo |
|---|---|
| Entrada | `data/processed/productoras_limpia.csv` (salida del notebook 01) |
| Salida | `data/processed/indicadores_kpi.csv`, `kpi_distritos.csv` y `ficha_comparativa.csv` (los usa el notebook 04) |

In [1]:
from pathlib import Path
import hashlib
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
OUT = ROOT / "data" / "processed"
pd.set_option("display.width", 220, "display.max_columns", 30, "display.max_colwidth", 70)

ARCHIVO = OUT / "productoras_limpia.csv"
base = pd.read_csv(ARCHIVO)
# Huella del archivo de entrada: permite comprobar que dos personas usan exactamente los mismos datos
print("Productoras:", len(base), "| huella SHA-256 de la entrada:", hashlib.sha256(ARCHIVO.read_bytes()).hexdigest()[:16], "…")

# Variable auxiliar: 1 si declara alguna dificultad para leer o escribir
base["sin_dificultad_lecto"] = (base["dificultad_lectoescritura"] == "NINGUNA").astype(int)

Productoras: 138 | huella SHA-256 de la entrada: fc256e59d625bb0e …


## 1. De la necesidad del negocio a los indicadores

Producers Direct administra un fondo rotatorio de crédito para productoras de café y debe decidir dónde y cómo acompañarlas. De los objetivos que compartió salen cuatro preguntas, y cada indicador responde a una de ellas.

| Pregunta | Qué necesita saber Producers Direct | Indicadores |
|---|---|---|
| **Conocer** | Quiénes son las productoras y qué acceso tienen | Productoras, celular, lectoescritura, capacitación, servicios financieros, socias titulares |
| **Decidir** | Dónde está el dinero del fondo y cómo se reparte | Monto solicitado, concentración por distrito, área y quintales |
| **Acompañar** | Qué carga representa el crédito para cada productora | Monto solicitado sobre ingreso |
| **Visitar** | Qué necesita saber quien visita a una productora | Posición de la productora frente a la típica de su cooperativa |

**Principio de lectura.** Un indicador siempre se acompaña de su denominador (de cuántas productoras sale) y no se compara sin contexto.

## 2. Definición de cada indicador

Cada indicador se define una sola vez, con su cálculo escrito en código. Así la definición y el resultado nunca se separan.

In [2]:
def pct(serie):
    """Porcentaje de unos entre las respuestas válidas; devuelve (valor, n que respondió)."""
    s = serie.dropna()
    return (100 * s.mean() if len(s) else np.nan), len(s)

INDICADORES = [
    # id, nombre, pregunta, fórmula, unidad, decisión, límite, función de cálculo
    ("productoras", "Productoras", "Conocer", "Conteo de códigos de productora", "productoras",
     "Dimensionar el grupo", "Una fila por productora", lambda d: (len(d), len(d))),
    ("monto_total", "Monto solicitado total", "Decidir", "Suma de monto_solicitado_pen", "S/",
     "Dimensionar el fondo", "Es lo solicitado: no prueba desembolso, deuda ni saldo", lambda d: (d["monto_solicitado_pen"].sum(), len(d))),
    ("monto_mediano", "Monto solicitado mediano", "Decidir", "Mediana de monto_solicitado_pen", "S/",
     "Saber cuánto pide una productora típica", "Pocos valores distintos", lambda d: (d["monto_solicitado_pen"].median(), len(d))),
    ("area_total", "Área de café total", "Decidir", "Suma de area_cafe_reportada", "unidad por confirmar",
     "Dimensionar la superficie", "La unidad no está confirmada; no se convierte", lambda d: (d["area_cafe_reportada"].sum(), len(d))),
    ("quintales_total", "Quintales acopiados totales", "Decidir", "Suma de quintales_acopiados_2025", "quintales",
     "Dimensionar el acopio", "Es lo entregado, no la cosecha total", lambda d: (d["quintales_acopiados_2025"].sum(), len(d))),
    ("quintales_mediana", "Quintales por productora (mediana)", "Conocer", "Mediana de quintales_acopiados_2025", "quintales",
     "Describir a la productora típica", "La media se distorsiona con fincas grandes", lambda d: (d["quintales_acopiados_2025"].median(), len(d))),
    ("smartphone", "Con smartphone propio", "Conocer", "Productoras con smartphone propio ÷ productoras que respondieron × 100", "%",
     "Elegir el canal de contacto", "Declarado por la productora", lambda d: pct(d["smartphone_propio"])),
    ("sin_dificultad", "Sin dificultad para leer ni escribir", "Conocer", "Productoras sin dificultad ÷ productoras × 100", "%",
     "Decidir si un material escrito alcanza", "Declarado, no medido con una prueba", lambda d: pct(d["sin_dificultad_lecto"])),
    ("capacitacion", "Con capacitación de la cooperativa", "Acompañar", "Productoras con capacitación en 2025 ÷ productoras × 100", "%",
     "Ver dónde reforzar la capacitación", "No mide frecuencia ni aprendizaje", lambda d: pct(d["capacitacion_cooperativa_2025"])),
    ("sin_servicios", "Sin ningún servicio financiero", "Conocer", "Productoras que declaran ninguno ÷ productoras que respondieron × 100", "%",
     "Planear educación financiera", "Excluye 1 respuesta contradictoria (Ninguno y Yape a la vez)", lambda d: pct(d["declara_ningun_servicio_financiero"])),
    ("titulares", "Socias titulares", "Conocer", "Productoras que son la socia titular ÷ productoras × 100", "%",
     "Saber a quién se dirige el trámite", "Titular no es sexo; la base no trae esa columna", lambda d: pct((d["parentesco_socio_titular"] == "TITULAR").astype(int))),
    ("monto_sobre_ingreso", "Monto solicitado sobre ingreso (mediana)", "Acompañar", "Mediana de monto_solicitado ÷ ingreso_principal_anual", "proporción",
     "Estimar la carga relativa del crédito", "No es servicio de deuda ni tasa de endeudamiento", lambda d: (d["monto_solicitado_sobre_ingreso_principal"].median(), len(d))),
    ("acopio_por_area", "Acopio por unidad de área (mediana)", "Visitar", "Mediana de quintales ÷ área reportada", "quintales por unidad de área",
     "Comparar fincas de distinto tamaño", "No es rendimiento agronómico; la unidad de área está por confirmar", lambda d: (d["acopio_por_unidad_area_2025"].median(), len(d))),
]
definiciones = pd.DataFrame([i[:7] for i in INDICADORES], columns=["id", "indicador", "responde a", "fórmula", "unidad", "decisión que apoya", "límite"]).set_index("id")
definiciones

,indicador,responde a,fórmula,unidad,decisión que apoya,límite
id,,,,,,
productoras,Productoras,Conocer,Conteo de códigos de productora,productoras,Dimensionar el grupo,Una fila por productora
monto_total,Monto solicitado total,Decidir,Suma de monto_solicitado_pen,S/,Dimensionar el fondo,"Es lo solicitado: no prueba desembolso, deuda ni saldo"
monto_mediano,Monto solicitado mediano,Decidir,Mediana de monto_solicitado_pen,S/,Saber cuánto pide una productora típica,Pocos valores distintos
area_total,Área de café total,Decidir,Suma de area_cafe_reportada,unidad por confirmar,Dimensionar la superficie,La unidad no está confirmada; no se convierte
quintales_total,Quintales acopiados totales,Decidir,Suma de quintales_acopiados_2025,quintales,Dimensionar el acopio,"Es lo entregado, no la cosecha total"
quintales_mediana,Quintales por productora (mediana),Conocer,Mediana de quintales_acopiados_2025,quintales,Describir a la productora típica,La media se distorsiona con fincas grandes
smartphone,Con smartphone propio,Conocer,Productoras con smartphone propio ÷ productoras que respondieron ×...,%,Elegir el canal de contacto,Declarado por la productora
sin_dificultad,Sin dificultad para leer ni escribir,Conocer,Productoras sin dificultad ÷ productoras × 100,%,Decidir si un material escrito alcanza,"Declarado, no medido con una prueba"
capacitacion,Con capacitación de la cooperativa,Acompañar,Productoras con capacitación en 2025 ÷ productoras × 100,%,Ver dónde reforzar la capacitación,No mide frecuencia ni aprendizaje


## 3. Cálculo por cooperativa

Cada indicador se calcula para el total y para cada cooperativa, siempre desde la misma tabla y con la misma función.

In [3]:
AMBITOS = {"Total": base, "Norandino": base[base["cooperativa"] == "NORANDINO"], "Bagua Grande": base[base["cooperativa"] == "BAGUA_GRANDE"]}
filas = []
for id_, nombre, pregunta, formula, unidad, decision, limite, calcular in INDICADORES:
    for ambito, d in AMBITOS.items():
        valor, n = calcular(d)
        filas.append({"id": id_, "indicador": nombre, "ámbito": ambito, "valor": valor, "n": n, "unidad": unidad})
kpi = pd.DataFrame(filas)
kpi.to_csv(OUT / "indicadores_kpi.csv", index=False, encoding="utf-8-sig")

ancho = kpi.pivot(index="indicador", columns="ámbito", values="valor")[["Total", "Norandino", "Bagua Grande"]]
ancho = ancho.loc[[i[1] for i in INDICADORES]]
ancho.round(2)

ámbito,Total,Norandino,Bagua Grande
indicador,,,
Productoras,138.00,113.00,25.00
Monto solicitado total,190175.00,154275.00,35900.00
Monto solicitado mediano,1375.00,1375.00,1500.00
Área de café total,310.36,255.86,54.50
Quintales acopiados totales,5444.00,4522.00,922.00
Quintales por productora (mediana),32.00,32.00,30.00
Con smartphone propio,86.23,88.50,76.00
Sin dificultad para leer ni escribir,81.16,76.99,100.00
Con capacitación de la cooperativa,86.23,85.84,88.00


## 4. Control de calidad de los indicadores

Antes de usar los resultados se comparan con cifras que se pueden verificar de forma independiente: los totales del original y las cifras que muestra el tablero publicado.

In [4]:
def valor(id_, ambito="Total"):
    return kpi[(kpi["id"] == id_) & (kpi["ámbito"] == ambito)]["valor"].iloc[0]

controles = {
    "Productoras": (valor("productoras"), 138),
    "Norandino": (valor("productoras", "Norandino"), 113),
    "Bagua Grande": (valor("productoras", "Bagua Grande"), 25),
    "Monto solicitado total (S/)": (valor("monto_total"), 190175),
    "Área de café total": (round(valor("area_total"), 2), 310.36),
    "Quintales acopiados totales": (valor("quintales_total"), 5444),
    "Sin servicios financieros, Bagua Grande (%)": (round(valor("sin_servicios", "Bagua Grande")), 29),
    "Socias titulares, total": (int((base["parentesco_socio_titular"] == "TITULAR").sum()), 49),
}
tabla = pd.DataFrame(controles, index=["calculado aquí", "cifra de control"]).T
tabla["coincide"] = np.isclose(tabla["calculado aquí"], tabla["cifra de control"])
assert tabla["coincide"].all(), "Un indicador no coincide con su cifra de control"
tabla

,calculado aquí,cifra de control,coincide
Productoras,138.00,138.00,True
Norandino,113.00,113.00,True
Bagua Grande,25.00,25.00,True
Monto solicitado total (S/),190175.00,190175.00,True
Área de café total,310.36,310.36,True
Quintales acopiados totales,5444.00,5444.00,True
"Sin servicios financieros, Bagua Grande (%)",29.00,29.00,True
"Socias titulares, total",49.00,49.00,True


**Lectura.** Todos los indicadores coinciden con las cifras de control. Los indicadores de este notebook y los del tablero salen de la misma base, con las mismas definiciones.

## 5. Dónde está el dinero: indicadores por distrito

Responde a la pregunta **Decidir**: ¿en qué distritos se concentra el monto solicitado?

In [5]:
dist = (base.groupby("distrito")
        .agg(productoras=("codigo_productora", "count"), monto=("monto_solicitado_pen", "sum"),
             area=("area_cafe_reportada", "sum"), quintales=("quintales_acopiados_2025", "sum"))
        .sort_values("monto", ascending=False))
dist["% del monto"] = 100 * dist["monto"] / dist["monto"].sum()
dist["% de las productoras"] = 100 * dist["productoras"] / dist["productoras"].sum()
dist["% acumulado del monto"] = dist["% del monto"].cumsum()
dist.round(1).to_csv(OUT / "kpi_distritos.csv", encoding="utf-8-sig")
dist.round(1)

,productoras,monto,area,quintales,% del monto,% de las productoras,% acumulado del monto
distrito,,,,,,,
Tabaconas,104,141900,235.2,4114,74.6,75.4,74.6
Bagua Grande,10,15800,25.2,439,8.3,7.2,82.9
San José del Alto,9,12375,20.7,408,6.5,6.5,89.4
Cajaruro,8,9800,13.5,266,5.2,5.8,94.6
Jamalca,5,7300,10.2,133,3.8,3.6,98.4
Copallín,2,3000,5.5,84,1.6,1.4,100.0


**Lectura.** Tabaconas concentra el 75 % del monto solicitado y también el 75 % de las productoras (104 de 138): su peso viene de cuántas productoras tiene, no de que cada una pida más. El resto de los distritos suma poco y varios tienen muy pocas productoras (Copallín tiene 2), de modo que sus porcentajes cambian mucho con una sola persona. **Decisión que apoya:** priorizar visitas y acompañamiento donde está la mayor parte del fondo (Tabaconas) y revisar con las cifras exactas antes de sacar conclusiones de los distritos pequeños.

## 6. Visitar: comparar a cada productora con la típica de su cooperativa

Para preparar una visita, un dato aislado dice poco: 40 quintales pueden ser mucho o poco según la cooperativa. Se calcula cuánto se aleja cada productora de la mediana de su cooperativa. La misma lógica usa la ficha de visita del tablero.

In [6]:
comparativa = base[["codigo_productora", "cooperativa", "distrito", "quintales_acopiados_2025", "ingreso_principal_pen_2025", "area_cafe_reportada"]].copy()
for var in ("quintales_acopiados_2025", "ingreso_principal_pen_2025", "area_cafe_reportada"):
    mediana_coop = base.groupby("cooperativa")[var].transform("median")
    comparativa[var + "_vs_tipica"] = (comparativa[var] / mediana_coop).round(2)   # 1 = igual a la típica
comparativa.to_csv(OUT / "ficha_comparativa.csv", index=False, encoding="utf-8-sig")

print("Cuántas productoras acopian el doble o más que la típica de su cooperativa:",
      int((comparativa["quintales_acopiados_2025_vs_tipica"] >= 2).sum()))
print("Cuántas acopian la mitad o menos:", int((comparativa["quintales_acopiados_2025_vs_tipica"] <= 0.5).sum()))
comparativa.sort_values("quintales_acopiados_2025_vs_tipica", ascending=False).head(5)

Cuántas productoras acopian el doble o más que la típica de su cooperativa: 15
Cuántas acopian la mitad o menos: 9


,codigo_productora,cooperativa,distrito,quintales_acopiados_2025,ingreso_principal_pen_2025,area_cafe_reportada,quintales_acopiados_2025_vs_tipica,ingreso_principal_pen_2025_vs_tipica,area_cafe_reportada_vs_tipica
26,PD-FR-027-FF-NOR,NORANDINO,Tabaconas,150,150000,5.57,4.69,5.00,2.78
23,PD-FR-024-FF-NOR,NORANDINO,Tabaconas,150,180000,5.00,4.69,6.00,2.50
121,PD-FR-122-FF-BG,BAGUA_GRANDE,Bagua Grande,110,60000,4.00,3.67,1.71,2.00
33,PD-FR-034-FF-NOR,NORANDINO,Tabaconas,111,100000,6.00,3.47,3.33,3.00
1,PD-FR-002-FF-NOR,NORANDINO,Tabaconas,109,100000,3.50,3.41,3.33,1.75


**Lectura.** La mayoría de las productoras está cerca de la típica de su cooperativa, y unas pocas la superan por mucho. En una visita esto permite decir "esta productora acopia casi 5 veces la mediana de su cooperativa" en lugar de un número suelto.

## 7. Resumen de resultados

Se leen siempre junto a su denominador (columna `n` de `indicadores_kpi.csv`).

In [7]:
resumen = ancho.copy()
resumen["unidad"] = [i[4] for i in INDICADORES]
resumen.round(2)

ámbito,Total,Norandino,Bagua Grande,unidad
indicador,,,,
Productoras,138.00,113.00,25.00,productoras
Monto solicitado total,190175.00,154275.00,35900.00,S/
Monto solicitado mediano,1375.00,1375.00,1500.00,S/
Área de café total,310.36,255.86,54.50,unidad por confirmar
Quintales acopiados totales,5444.00,4522.00,922.00,quintales
Quintales por productora (mediana),32.00,32.00,30.00,quintales
Con smartphone propio,86.23,88.50,76.00,%
Sin dificultad para leer ni escribir,81.16,76.99,100.00,%
Con capacitación de la cooperativa,86.23,85.84,88.00,%


## 8. Conclusiones

- **Conocer.** 86 % tiene smartphone, 81 % no declara dificultad para leer o escribir y 77 % no usa ningún servicio financiero. Las diferencias entre cooperativas están en servicios financieros y lectoescritura, no en capacitación.
- **Decidir.** El fondo solicitado suma S/ 190,175 para 138 productoras; tres cuartas partes están en Tabaconas.
- **Acompañar.** El monto solicitado equivale a un 4 % mediano del ingreso anual declarado de la productora; es una carga relativa baja, aunque no mide el servicio real de la deuda.
- **Visitar.** Comparar con la productora típica de su cooperativa da un punto de partida concreto para la conversación.

**Límites.** Los indicadores describen lo declarado en un solo corte. No miden mora, desembolsos ni rendimiento, y el área y el quintal siguen sin unidad confirmada.

**Siguiente:** `04_dashboard.ipynb` construye los gráficos que lleva el tablero a partir de `indicadores_kpi.csv` y `kpi_distritos.csv`.